# Milestone 4 - Multiple-Choice Classification and LoRA Fine-Tuning

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, TaskType, get_peft_model
from datasets import Dataset

# Kaggle path with local fallback
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
import os
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'train.csv'

train = pd.read_csv(DATA_PATH)
print(f'Loaded train.csv: {len(train)} rows')

# A=0, B=1, C=2, D=3, E=4
ANSWER_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
ID2ANSWER = {v: k for k, v in ANSWER_MAP.items()}
train['label'] = train['answer'].map(ANSWER_MAP)

# Tokenizer shared by every tokenization question
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print(f'Tokenizer: bert-base-uncased, vocab_size={tokenizer.vocab_size}')
print(f'GPU available: {torch.cuda.is_available()}')

Loaded train.csv: 2000 rows


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: bert-base-uncased, vocab_size=30522
GPU available: True


In [2]:
def format_choice(prompt_text, option_text):
    """Q2 format: prompt + ' [SEP] ' + option."""
    return str(prompt_text) + ' [SEP] ' + str(option_text)


def tokenize_row(row, max_length=128):
    """
    Tokenize one row's 5 (prompt, option) pairs and return
    input_ids / attention_mask of shape (5, max_length) plus the int label.
    """
    prompt = str(row['prompt'])
    options = [str(row[opt]) for opt in 'ABCDE']
    enc = tokenizer(
        [prompt] * 5, options,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )
    return {
        'input_ids': enc['input_ids'],         # (5, max_length)
        'attention_mask': enc['attention_mask'],
        'labels': int(row['label']),
    }

# Quick sanity check on row 0
sample = tokenize_row(train.iloc[0])
print(f'Sample input_ids shape: {tuple(sample["input_ids"].shape)}')
print(f'Sample attention_mask shape: {tuple(sample["attention_mask"].shape)}')
print(f'Sample label: {sample["labels"]}')

Sample input_ids shape: (5, 128)
Sample attention_mask shape: (5, 128)
Sample label: 1


---

## Q1. Encoded Label for Row Index 150

**Task**: Encode the `answer` column with `A=0, B=1, C=2, D=3, E=4`. Report the
encoded label for the row at index 150.

In [3]:
row_150 = train.iloc[150]
q1_answer = int(row_150['label'])

print(f'Row 150 answer letter : {row_150["answer"]}')
print(f'Row 150 encoded label : {q1_answer}')
print()
print(f'ANSWER Q1: {q1_answer}')

Row 150 answer letter : C
Row 150 encoded label : 2

ANSWER Q1: 2


---

## Q2. Character Length of the Formatted Option B String (Row 0)

**Task**: For row index 0, build the formatted input
`str(prompt) + " [SEP] " + str(option_B)`. Report the exact character length
of that string using Python's `len()`.

In [4]:
row_0 = train.iloc[0]
formatted_b = format_choice(row_0['prompt'], row_0['B'])

print(f'Prompt (first 80)   : {str(row_0["prompt"])[:80]}...')
print(f'Option B (first 80) : {str(row_0["B"])[:80]}...')
print(f'Formatted (first 100): {formatted_b[:100]}...')
print()

q2_answer = len(formatted_b)
print(f'ANSWER Q2: {q2_answer}')

Prompt (first 80)   : Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
Option B (first 80) : Martin Heidegger believes that humans do not exist inside time, but that they ar...
Formatted (first 100): Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and ...

ANSWER Q2: 407


---

## Q3. Second Dimension of the Single-Row MCQ Tensor

**Task**: Tokenize the 5 formatted inputs for row 0 with `padding='max_length'`,
`truncation=True`, `max_length=128`, `return_tensors='pt'`. Reshape to
multiple-choice format. The final `input_ids` has shape `[1, 5, 128]`. Report
the value of the second dimension.

The second dimension is the number of choices per question, which is fixed at
5 for this competition.

In [5]:
# Tokenize the 5 (prompt, option) pairs for row 0
row_0_tokenized = tokenize_row(train.iloc[0], max_length=128)

# Add the batch dimension to get multiple-choice shape: (1, 5, 128)
input_ids_mc = row_0_tokenized['input_ids'].unsqueeze(0)
print(f'input_ids shape after unsqueeze: {tuple(input_ids_mc.shape)}')

q3_answer = input_ids_mc.shape[1]
print(f'Second dimension value: {q3_answer}')
print()
print(f'ANSWER Q3: {q3_answer}')


input_ids shape after unsqueeze: (1, 5, 128)
Second dimension value: 5

ANSWER Q3: 5


---

## Q4. Total Token Positions in a Batch of 16 Rows

**Task**: Tokenize the first 16 rows of `train.csv` as multiple-choice
examples (5 choices per row, each choice padded to length 128). The final
`input_ids` has shape `[16, 5, 128]`. Report the total number of token
positions in this tensor.

In [6]:
# The shape is given as [16, 5, 128], so the total is the product
q4_answer = 16 * 5 * 128

print(f'Shape: [16, 5, 128]')
print(f'Total token positions: 16 x 5 x 128 = {q4_answer}')
print()

# Verification - actually build the tensor
batch_input_ids = torch.stack([
    tokenize_row(train.iloc[i], max_length=128)['input_ids']
    for i in range(16)
])
print(f'Verification - batch input_ids shape: {tuple(batch_input_ids.shape)}')
print(f'Verification - total positions      : {batch_input_ids.numel()}')
print()
print(f'ANSWER Q4: {q4_answer}')

Shape: [16, 5, 128]
Total token positions: 16 x 5 x 128 = 10240

Verification - batch input_ids shape: (16, 5, 128)
Verification - total positions      : 10240

ANSWER Q4: 10240


---

## Q5. Number of Logits per Question

**Task**: Load `bert-base-uncased` via `AutoModelForMultipleChoice`. Tokenize
row 0 as 5 choices and forward through the model. The output logits have shape
`[1, 5]`. Report how many logits are produced for one question.

In [7]:
model_mc = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
model_mc.eval()

# Build the (1, 5, 128) input from row 0
inputs_q5 = {
    'input_ids': row_0_tokenized['input_ids'].unsqueeze(0),
    'attention_mask': row_0_tokenized['attention_mask'].unsqueeze(0),
}

with torch.no_grad():
    outputs_q5 = model_mc(**inputs_q5)

print(f'Logits shape: {tuple(outputs_q5.logits.shape)}')
q5_answer = outputs_q5.logits.shape[1]
print(f'Number of logits per question: {q5_answer}')
print()
print(f'ANSWER Q5: {q5_answer}')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: (1, 5)
Number of logits per question: 5

ANSWER Q5: 5


---

## Q6. Dimensionality of the Supervised Loss Tensor

**Task**: For row 0, pass the tokenized 5-choice input **and** the correct
encoded label into `AutoModelForMultipleChoice`. The model returns a scalar
loss. Report the number of dimensions of that loss tensor.


In [8]:
# Forward with labels - the model computes the loss internally
labels_q6 = torch.tensor([int(row_0['label'])])

with torch.no_grad():
    outputs_q6 = model_mc(**inputs_q5, labels=labels_q6)

loss_tensor = outputs_q6.loss
print(f'Loss value : {loss_tensor.item()}')
print(f'Loss shape : {loss_tensor.shape}')
print(f'Loss ndim  : {loss_tensor.dim()}')

q6_answer = loss_tensor.dim()
print()
print(f'ANSWER Q6: {q6_answer}')

# Free the base MC model before Q7 (we will use a LoRA-wrapped copy instead)
del model_mc
import gc; gc.collect()

Loss value : 1.6003844738006592
Loss shape : torch.Size([])
Loss ndim  : 0

ANSWER Q6: 0


119

---

## Q7. LoRA Trainable Parameter Count

**Task**: Apply LoRA to `bert-base-uncased` loaded as a multiple-choice model
with this exact config:

- `r = 8`
- `lora_alpha = 16`
- `target_modules = ["query", "value"]`
- `lora_dropout = 0.1`
- `bias = "none"`
- `task_type = TaskType.SEQ_CLS`

Count parameters where `requires_grad=True`.

In [9]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)

# Fresh MC model + LoRA wrapper
model_lora_base = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
model_lora = get_peft_model(model_lora_base, lora_config)

trainable_params = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_lora.parameters())
pct = 100 * trainable_params / total_params

print(f'Total params     : {total_params}')
print(f'Trainable params : {trainable_params}  ({pct:.3f}% of total)')

q7_answer = trainable_params
print()
print(f'ANSWER Q7: {q7_answer}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params     : 109778690
Trainable params : 295681  (0.269% of total)

ANSWER Q7: 295681


---

## Q8. Tokenized Choices Stored in `input_ids` (HF Dataset)

**Task**: Build a Hugging Face `Dataset` from the first 100 rows of
`train.csv`. For each row, store `input_ids` with shape `[5, 128]`,
`attention_mask` with shape `[5, 128]`, and `labels` as the encoded answer
label. For the first dataset item, `input_ids` has shape `[5, 128]`. How many
tokenized choices are stored in `input_ids`?

In [10]:
# Build the HF Dataset from the first 100 rows
first_100 = train.head(100)
ds = Dataset.from_pandas(first_100[['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'label']])

# Map each row to {input_ids, attention_mask, labels} via the helper
# The helper returns torch tensors - HF Dataset will store them as lists of lists
ds = ds.map(tokenize_row, remove_columns=['id', 'prompt', 'A', 'B', 'C', 'D', 'E'])

print(f'Dataset size: {len(ds)}')
print(f'First item input_ids num_choices: {len(ds[0]["input_ids"])}')
print(f'First item input_ids seq_len    : {len(ds[0]["input_ids"][0])}')

q8_answer = len(ds[0]['input_ids'])
print()
print(f'ANSWER Q8: {q8_answer}')

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset size: 100
First item input_ids num_choices: 5
First item input_ids seq_len    : 128

ANSWER Q8: 5


---

## Q9. Tiny LoRA Fine-Tuning - Final `global_step`

**Task**: Fine-tune a LoRA multiple-choice model on the first 32 rows using
Hugging Face `Trainer` with:

- `max_length = 64`
- `per_device_train_batch_size = 4`
- `gradient_accumulation_steps = 1`
- `max_steps = 4`

Report the final `global_step` reported by the `Trainer`.

In [11]:
import transformers
print(transformers.__version__)

5.0.0


In [12]:
# Build a fine-tune dataset of the first 32 rows with max_length=64
first_32 = train.head(32)
ds_ft = Dataset.from_pandas(first_32[['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'label']])

def tokenize_row_short(row):
    return tokenize_row(row, max_length=64)

ds_ft = ds_ft.map(tokenize_row_short, remove_columns=['id', 'prompt', 'A', 'B', 'C', 'D', 'E'])
ds_ft.set_format('torch')
print(f'Fine-tune dataset size: {len(ds_ft)}')

# Fresh LoRA model for fine-tuning (we keep the Q7 model untouched for comparison)
model_ft_base = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
model_ft = get_peft_model(model_ft_base, lora_config)

training_args = TrainingArguments(
    output_dir="./milestone4_q9_outputs",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model_ft,
    args=training_args,
    train_dataset=ds_ft,
)

train_result = trainer.train()
q9_answer = int(train_result.global_step)
print()
print(f'Final global_step: {q9_answer}')
print()
print(f'ANSWER Q9: {q9_answer}')


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Fine-tune dataset size: 32


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
1,3.181635
2,3.193148
3,3.392499
4,3.318687



Final global_step: 4

ANSWER Q9: 4


---

## Q10. Probability of Option E After Fine-Tuning (Row 0)

**Task**: Using the fine-tuned LoRA model from Q9, run inference on row 0 with
`max_length=64` and apply softmax to the logits. Report the probability
assigned to option E. Round to 4 decimal places.

In [13]:

row_0_short = tokenize_row(train.iloc[0], max_length=64)

device = next(model_ft.parameters()).device

inputs_q10 = {
    "input_ids": row_0_short["input_ids"].unsqueeze(0).to(device),
    "attention_mask": row_0_short["attention_mask"].unsqueeze(0).to(device),
}

model_ft.eval()

with torch.no_grad():
    outputs = model_ft(**inputs_q10)

probs = torch.softmax(outputs.logits[0], dim=0)

q10_answer = round(probs[4].item(), 4)

print(f"ANSWER Q10: {q10_answer}")

ANSWER Q10: 0.2153
